# Dataset Unification & Mapping

**Purpose**: Verify all datasets have identical structure, then merge and map them into a single unified dataset.

**Prerequisites**: Run `01_dataset_pipeline_local.ipynb` first


## Section 1: Imports & Load Data


In [80]:
import os
import json
from pathlib import Path
from collections import Counter
from typing import List, Dict, Any

In [81]:
# Configure paths (works in both Colab and local)
try:
    from google.colab import drive
    # Running in Colab - use Google Drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    IS_COLAB = True
    print("✓ Running in Colab - using Google Drive paths")
except ImportError:
    # Running locally
    BASE_DIR = Path("/Users/anas/Projects/code-security-identifier")
    IS_COLAB = False
    print("✓ Running locally - using local paths")

DATASETS_DIR = BASE_DIR / "datasets"


def read_jsonl(path):
    """Read JSONL file into list of records."""
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f]


def write_jsonl(path, records):
    """Write list of records to JSONL file."""
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


# Load both splits
train = read_jsonl(DATASETS_DIR / "FINAL_train.jsonl")
val = read_jsonl(DATASETS_DIR / "FINAL_val.jsonl")

print(f"✓ Loaded FINAL_train.jsonl: {len(train):,} records")
print(f"✓ Loaded FINAL_val.jsonl: {len(val):,} records")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Running in Colab - using Google Drive paths
✓ Loaded FINAL_train.jsonl: 3,677 records
✓ Loaded FINAL_val.jsonl: 408 records


## Section 2: Verify Identical Structure


In [82]:
print("Verifying structure across all datasets...\n")

# Required fields every record must have
REQUIRED_FIELDS = ["lines", "raw_lines", "label", "type", "cwe_id", "dataset_source"]


def get_structure(record: Dict) -> Dict:
    """
    Extract structure info from a record.
    Returns field names, types, and array lengths.
    """
    return {
        "fields": set(record.keys()),
        "field_types": {k: type(v).__name__ for k, v in record.items()},
        "array_lengths": {
            k: len(v) if isinstance(v, (list, dict)) else None
            for k, v in record.items()
        },
    }


def validate_records(data: List[Dict], split_name: str) -> bool:
    """
    Validate all records in a split have correct structure.
    Returns True if valid, False otherwise.
    """
    issues = []

    for i, record in enumerate(data):
        # Check required fields exist
        missing = [f for f in REQUIRED_FIELDS if f not in record]
        if missing:
            issues.append(f"  [{i}] missing fields: {missing}")
            continue

        # Check array field lengths match
        n_labels = len(record["label"])
        n_lines = len(record["lines"])
        n_raw = len(record["raw_lines"])
        n_types = len(record["type"])

        if not (n_labels == n_lines == n_raw == n_types):
            issues.append(
                f"  [{i}] array mismatch: label={n_labels}, "
                f"lines={n_lines}, raw_lines={n_raw}, type={n_types}"
            )

        # Check label values are 0 or 1
        invalid_labels = [l for l in record["label"] if l not in (0, 1)]
        if invalid_labels:
            issues.append(f"  [{i}] invalid label values: {set(invalid_labels)}")

        # Check CWE format
        cwe = record["cwe_id"]
        if not (isinstance(cwe, str) and (cwe.startswith("CWE-") or cwe == "unknown")):
            issues.append(f"  [{i}] invalid CWE format: {cwe}")

        # Check dataset_source is string
        if not isinstance(record["dataset_source"], str):
            issues.append(
                f"  [{i}] dataset_source not string: {type(record['dataset_source'])}"
            )

    if issues:
        print(f"❌ {split_name}: Found {len(issues)} issues")
        for issue in issues[:5]:
            print(issue)
        if len(issues) > 5:
            print(f"  ... and {len(issues) - 5} more")
        return False
    else:
        print(f"✓ {split_name}: All {len(data):,} records have valid structure")
        return True


# Validate both splits
train_valid = validate_records(train, "FINAL_train")
val_valid = validate_records(val, "FINAL_val")

if train_valid and val_valid:
    print("\n✓✓✓ ALL DATASETS HAVE IDENTICAL STRUCTURE ✓✓✓")
else:
    print("\n⚠ Structure issues found. Please fix before unifying.")

Verifying structure across all datasets...

✓ FINAL_train: All 3,677 records have valid structure
✓ FINAL_val: All 408 records have valid structure

✓✓✓ ALL DATASETS HAVE IDENTICAL STRUCTURE ✓✓✓


## Section 3: Map and Normalize Records


In [83]:
def normalize_record(record: Dict, split_origin: str, record_id: int) -> Dict:
    """
    Normalize and enhance a record with mapping metadata.

    Adds:
    - record_id: unique ID within split
    - split_origin: which split this came from (train/val)
    - global_id: unique ID across all datasets
    - num_statements: total statements in this record
    - num_vulnerable: total vulnerable statements
    """

    num_stmts = len(record["label"])
    num_vuln = sum(record["label"])

    # Add metadata
    record["record_id"] = record_id
    record["split_origin"] = split_origin
    record["num_statements"] = num_stmts
    record["num_vulnerable"] = num_vuln
    record["is_vulnerable"] = num_vuln > 0

    return record


print("Normalizing records...")

# Normalize train split
normalized_train = []
for i, record in enumerate(train):
    normalized = normalize_record(record.copy(), "train", i)
    normalized_train.append(normalized)

# Normalize val split
normalized_val = []
for i, record in enumerate(val):
    normalized = normalize_record(record.copy(), "val", i)
    normalized_val.append(normalized)

print(f"✓ Normalized {len(normalized_train):,} train records")
print(f"✓ Normalized {len(normalized_val):,} val records")

Normalizing records...
✓ Normalized 3,677 train records
✓ Normalized 408 val records


## Section 4: Create Unified Dataset


In [84]:
print("Creating unified dataset...\n")

# Merge both splits
unified_records = normalized_train + normalized_val

# Add global IDs
for global_id, record in enumerate(unified_records):
    record["global_id"] = global_id

print(f"✓ Combined {len(unified_records):,} total records")
print(f"  - Train: {len(normalized_train):,} records")
print(f"  - Val:   {len(normalized_val):,} records")

# Statistics
total_stmts = sum(r["num_statements"] for r in unified_records)
total_vuln = sum(r["num_vulnerable"] for r in unified_records)
total_vuln_funcs = sum(1 for r in unified_records if r["is_vulnerable"])

print(f"\n✓ Statistics:")
print(f"  - Total statements: {total_stmts:,}")
print(f"  - Vulnerable statements: {total_vuln:,} ({total_vuln/total_stmts*100:.1f}%)")
print(
    f"  - Vulnerable functions: {total_vuln_funcs:,} ({total_vuln_funcs/len(unified_records)*100:.1f}%)"
)

Creating unified dataset...

✓ Combined 4,085 total records
  - Train: 3,677 records
  - Val:   408 records

✓ Statistics:
  - Total statements: 288,591
  - Vulnerable statements: 25,181 (8.7%)
  - Vulnerable functions: 2,474 (60.6%)


## Section 5: Create Cross-References & Mappings


In [85]:
print("Creating mapping indices...\n")

# Create various indices for quick lookups
mappings = {
    "by_split": {"train": [], "val": []},
    "by_source": {},
    "by_cwe": {},
    "vulnerable_indices": [],
    "safe_indices": [],
}

for record in unified_records:
    gid = record["global_id"]
    split = record["split_origin"]
    source = record["dataset_source"]
    cwe = record["cwe_id"]
    is_vuln = record["is_vulnerable"]

    # By split
    mappings["by_split"][split].append(gid)

    # By source
    if source not in mappings["by_source"]:
        mappings["by_source"][source] = []
    mappings["by_source"][source].append(gid)

    # By CWE
    if cwe not in mappings["by_cwe"]:
        mappings["by_cwe"][cwe] = []
    mappings["by_cwe"][cwe].append(gid)

    # By vulnerability status
    if is_vuln:
        mappings["vulnerable_indices"].append(gid)
    else:
        mappings["safe_indices"].append(gid)

# Print mapping summary
print("Mapping Summary:")
print(f"\n  By Split:")
for split, indices in mappings["by_split"].items():
    print(f"    {split}: {len(indices):,} records")

print(f"\n  By Source:")
for source, indices in sorted(mappings["by_source"].items()):
    print(f"    {source}: {len(indices):,} records")

print(f"\n  By CWE (top 10):")
for cwe, indices in sorted(mappings["by_cwe"].items(), key=lambda x: -len(x[1]))[:10]:
    print(f"    {cwe}: {len(indices):,} records")

print(f"\n  By Vulnerability:")
print(f"    Vulnerable: {len(mappings['vulnerable_indices']):,} records")
print(f"    Safe:       {len(mappings['safe_indices']):,} records")

Creating mapping indices...

Mapping Summary:

  By Split:
    train: 3,677 records
    val: 408 records

  By Source:
    funclevel: 2,718 records
    securityeval: 121 records
    vudenc: 1,246 records

  By CWE (top 10):
    unknown: 2,036 records
    CWE-089: 589 records
    CWE-079: 404 records
    CWE-022: 232 records
    CWE-601: 209 records
    CWE-077: 194 records
    CWE-352: 181 records
    CWE-094: 122 records
    CWE-611: 6 records
    CWE-020: 6 records

  By Vulnerability:
    Vulnerable: 2,474 records
    Safe:       1,611 records


## Section 6: Save Unified Dataset


In [86]:
print("Saving unified dataset...\n")

# Save unified JSONL
unified_path = DATASETS_DIR / "UNIFIED.jsonl"
write_jsonl(unified_path, unified_records)
size_mb = os.path.getsize(unified_path) / (1024 * 1024)
print(f"✓ Saved UNIFIED.jsonl: {len(unified_records):,} records ({size_mb:.1f} MB)")

# Save mappings as JSON
mappings_path = DATASETS_DIR / "UNIFIED_mappings.json"
with open(mappings_path, "w") as f:
    json.dump(mappings, f, indent=2)
print(f"✓ Saved UNIFIED_mappings.json: cross-reference indices")

# Save metadata about unified dataset
metadata = {
    "total_records": len(unified_records),
    "total_statements": total_stmts,
    "total_vulnerable_statements": total_vuln,
    "total_vulnerable_functions": total_vuln_funcs,
    "vulnerable_percentage": total_vuln / total_stmts * 100 if total_stmts > 0 else 0,
    "splits": {
        "train": {"records": len(normalized_train)},
        "val": {"records": len(normalized_val)},
    },
    "sources": {src: len(ids) for src, ids in mappings["by_source"].items()},
    "cwes": {cwe: len(ids) for cwe, ids in mappings["by_cwe"].items()},
    "required_fields": REQUIRED_FIELDS,
}

metadata_path = DATASETS_DIR / "UNIFIED_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Saved UNIFIED_metadata.json: dataset metadata")

print(f"\n✓ All files saved to: {DATASETS_DIR}")

Saving unified dataset...

✓ Saved UNIFIED.jsonl: 4,085 records (29.0 MB)
✓ Saved UNIFIED_mappings.json: cross-reference indices
✓ Saved UNIFIED_metadata.json: dataset metadata

✓ All files saved to: /content/drive/MyDrive/CSI_Project/datasets


## Section 7: Verify Unified Dataset


In [87]:
print("\n" + "=" * 70)
print("UNIFIED DATASET VERIFICATION")
print("=" * 70)

# Reload and verify
unified_loaded = read_jsonl(unified_path)
print(f"\n✓ Reloaded {len(unified_loaded):,} records from UNIFIED.jsonl")

# Check first few records have all required fields
print(f"\nSample record structure:")
sample = unified_loaded[0]
for key in [
    "global_id",
    "record_id",
    "split_origin",
    "dataset_source",
    "cwe_id",
    "num_statements",
    "num_vulnerable",
    "is_vulnerable",
]:
    value = sample.get(key, "N/A")
    print(f"  {key:>20}: {value}")

# Verify unification
verify_issues = []
for record in unified_loaded:
    if "global_id" not in record:
        verify_issues.append(f"missing global_id")
    if "split_origin" not in record:
        verify_issues.append(f"missing split_origin")
    if record["split_origin"] not in ["train", "val"]:
        verify_issues.append(f"invalid split_origin: {record['split_origin']}")

if verify_issues:
    print(f"\n⚠ Found {len(set(verify_issues))} verification issues")
else:
    print(f"\n✓ All records have mapping metadata")
    print(f"✓ All records have correct split_origin")
    print(f"✓ Unified dataset is valid and ready to use")

print(f"\n" + "=" * 70)


UNIFIED DATASET VERIFICATION

✓ Reloaded 4,085 records from UNIFIED.jsonl

Sample record structure:
             global_id: 0
             record_id: 0
          split_origin: train
        dataset_source: funclevel
                cwe_id: unknown
        num_statements: 17
        num_vulnerable: 1
         is_vulnerable: True

✓ All records have mapping metadata
✓ All records have correct split_origin
✓ Unified dataset is valid and ready to use



## Section 8: Usage Examples


In [88]:
print("\nUsage Examples:\n")

print("1. Load unified dataset:")
print("   unified = read_jsonl('datasets/UNIFIED.jsonl')")
print()

print("2. Get all records from train split:")
print(f"   train_records = [r for r in unified if r['split_origin'] == 'train']")
print(
    f"   # Returns {len([r for r in unified_loaded if r['split_origin'] == 'train']):,} records"
)
print()

print("3. Get all vulnerable records:")
print(f"   vulnerable = [r for r in unified if r['is_vulnerable']]")
print(
    f"   # Returns {len([r for r in unified_loaded if r['is_vulnerable']]):,} records"
)
print()

print("4. Get records by source:")
print(f"   vudenc = [r for r in unified if r['dataset_source'] == 'vudenc']")
print()

print("5. Get records by CWE:")
print(f"   cwe_089 = [r for r in unified if r['cwe_id'] == 'CWE-089']")
print()

print("6. Use mapping indices:")
print(f"   with open('datasets/UNIFIED_mappings.json') as f:")
print(f"       mappings = json.load(f)")
print(f"   # Get train indices: mappings['by_split']['train']")
print(f"   # Get vulnerable indices: mappings['vulnerable_indices']")


Usage Examples:

1. Load unified dataset:
   unified = read_jsonl('datasets/UNIFIED.jsonl')

2. Get all records from train split:
   train_records = [r for r in unified if r['split_origin'] == 'train']
   # Returns 3,677 records

3. Get all vulnerable records:
   vulnerable = [r for r in unified if r['is_vulnerable']]
   # Returns 2,474 records

4. Get records by source:
   vudenc = [r for r in unified if r['dataset_source'] == 'vudenc']

5. Get records by CWE:
   cwe_089 = [r for r in unified if r['cwe_id'] == 'CWE-089']

6. Use mapping indices:
   with open('datasets/UNIFIED_mappings.json') as f:
       mappings = json.load(f)
   # Get train indices: mappings['by_split']['train']
   # Get vulnerable indices: mappings['vulnerable_indices']


## Section 9: Model: CWE classification head (7 classes)

Defines a lightweight classifier head for 7 CWE classes used in your current dataset pipeline.

In [89]:
import torch
import torch.nn as nn

# Fixed 7-class target space requested for CWE classification
CWE_7_CLASSES = [
    "CWE-077",  # command injection
    "CWE-601",  # open redirect
    "CWE-022",  # path traversal/disclosure
    "CWE-094",  # code injection / RCE
    "CWE-089",  # SQL injection
    "CWE-352",  # CSRF
    "CWE-079",  # XSS
]

CWE_TO_INDEX = {cwe: i for i, cwe in enumerate(CWE_7_CLASSES)}
INDEX_TO_CWE = {i: cwe for cwe, i in CWE_TO_INDEX.items()}


def map_cwe_to_target(cwe_id: str):
    """Maps raw CWE IDs to 0..6 labels; returns -1 when not in the 7-class subset."""
    return CWE_TO_INDEX.get(cwe_id, -1)


class CWEClassificationHead(nn.Module):
    """# Model: CWE classification head (7 classes)"""

    def __init__(self, hidden_size: int, num_classes: int = 7, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, pooled_features: torch.Tensor) -> torch.Tensor:
        x = self.dropout(pooled_features)
        return self.classifier(x)


print("Defined CWEClassificationHead with 7 classes")
print("Class mapping:")
for idx, cwe in INDEX_TO_CWE.items():
    print(f"  {idx}: {cwe}")

Defined CWEClassificationHead with 7 classes
Class mapping:
  0: CWE-077
  1: CWE-601
  2: CWE-022
  3: CWE-094
  4: CWE-089
  5: CWE-352
  6: CWE-079


## Section 10: Model: LoRA wrapper for GraphCodeBERT

Wraps `microsoft/graphcodebert-base` with PEFT LoRA and attaches the 7-class CWE head.

In [90]:
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model


def count_trainable_params(model: nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = 100 * trainable / total if total else 0.0
    return trainable, total, pct


class GraphCodeBERTLoRACWEModel(nn.Module):
    """# Model: LoRA wrapper for GraphCodeBERT"""

    def __init__(
        self,
        model_name: str = "microsoft/graphcodebert-base",
        num_cwe_classes: int = 7,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        # For RoBERTa-family attention modules, query/value are standard LoRA targets.
        peft_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["query", "value"],
            bias="none",
        )
        self.encoder = get_peft_model(self.encoder, peft_cfg)

        hidden_size = self.encoder.config.hidden_size
        self.cwe_head = CWEClassificationHead(
            hidden_size=hidden_size, num_classes=num_cwe_classes
        )
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-1)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = out.last_hidden_state[:, 0, :]  # [CLS] token representation

        logits = self.cwe_head(cls_repr)
        result = {"logits": logits}

        if labels is not None:
            result["loss"] = self.loss_fn(logits, labels)

        return result


# Smoke-test instantiation
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "microsoft/graphcodebert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = GraphCodeBERTLoRACWEModel(model_name=model_name).to(device)

sample_code = [
    "def f(user): return 'SELECT * FROM users WHERE name=' + user",
    "def add(a, b): return a + b",
]
encoded = tokenizer(
    sample_code, padding=True, truncation=True, max_length=256, return_tensors="pt"
)
encoded = {k: v.to(device) for k, v in encoded.items()}

with torch.no_grad():
    pred = model(
        input_ids=encoded["input_ids"], attention_mask=encoded["attention_mask"]
    )

print(f"Logits shape: {tuple(pred['logits'].shape)}")  # expected: (batch_size, 7)
trainable, total, pct = count_trainable_params(model)
print(f"Trainable params: {trainable:,} / {total:,} ({pct:.2f}%)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: (2, 7)
Trainable params: 300,295 / 124,945,927 (0.24%)
